# Cancer-Gate CG9-005 경량 검증
**SMILES:** OC(=O)C1(c2ccc(F)c(F)c2)C(F)(F)C1CN1CC(F)(F)CCC1
**타깃:** GRM4 (PDB: 7E9H)
**목표:** MM-GBSA 보정 + MD 10ns RMSD 확인

In [ ]:
# Step 0 — 패키지 설치
!pip install rdkit openmm openmmforcefields openff-toolkit mdanalysis matplotlib numpy -q
!conda install -c conda-forge ambertools -y -q 2>/dev/null || echo 'AmberTools 스킵 — OpenMM 방식으로 진행'
print('설치 완료')

In [ ]:
# Step 1 — RDKit 기본 검증
from rdkit import Chem
from rdkit.Chem import Descriptors, QED, AllChem
import numpy as np

SMILES = 'OC(=O)C1(c2ccc(F)c(F)c2)C(F)(F)C1CN1CC(F)(F)CCC1'
mol = Chem.MolFromSmiles(SMILES)

mw   = Descriptors.MolWt(mol)
logp = Descriptors.MolLogP(mol)
qed  = QED.qed(mol)
hbd  = Descriptors.NumHDonors(mol)
hba  = Descriptors.NumHAcceptors(mol)
tpsa = Descriptors.TPSA(mol)

print('=== CG9-005 RDKit 검증 ===')
print(f'MW:   {mw:.1f} Da')
print(f'LogP: {logp:.3f}')
print(f'QED:  {qed:.3f}')
print(f'HBD/HBA: {hbd}/{hba}')
print(f'TPSA: {tpsa:.1f}')
print(f'Lipinski: {"PASS" if mw<500 and logp<5 and hbd<=5 and hba<=10 else "FAIL"}')

In [ ]:
# Step 2 — PDB 다운로드 및 준비
import urllib.request
import os

pdb_id = '7E9H'
url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
urllib.request.urlretrieve(url, f'{pdb_id}.pdb')
print(f'{pdb_id}.pdb 다운로드 완료')

# 체인 R 추출 (GRM4)
with open(f'{pdb_id}.pdb') as f:
    lines = f.readlines()
chain_r = [l for l in lines if l.startswith('ATOM') and l[21]=='R']
with open('GRM4_chainR.pdb', 'w') as f:
    f.writelines(chain_r)
    f.write('END\n')
print(f'Chain R 추출: {len(chain_r)}개 원자')

In [ ]:
# Step 3 — OpenMM MD 10ns (경량)
from openmm.app import *
from openmm import *
from openmm.unit import *
from openmmforcefields.generators import GAFFTemplateGenerator
from openff.toolkit.topology import Molecule
import mdanalysis as mda
import matplotlib.pyplot as plt

print('OpenMM MD 준비 중...')

# 리간드 준비
smiles = 'OC(=O)C1(c2ccc(F)c(F)c2)C(F)(F)C1CN1CC(F)(F)CCC1'
ligand = Molecule.from_smiles(smiles)
gaff = GAFFTemplateGenerator(molecules=[ligand])

# 단백질 로드
pdb = PDBFile('GRM4_chainR.pdb')
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
forcefield.registerTemplateGenerator(gaff.generator)

# 시스템 구성
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addSolvent(forcefield, model='tip3p', padding=10*angstroms)
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometers,
    constraints=HBonds
)

# 인테그레이터
integrator = LangevinMiddleIntegrator(310*kelvin, 1/picosecond, 0.002*picoseconds)
platform = Platform.getPlatformByName('CUDA')
simulation = Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)

# 에너지 최소화
print('에너지 최소화 중...')
simulation.minimizeEnergy()

# 평형화 100ps
print('평형화 100ps...')
simulation.context.setVelocitiesToTemperature(310*kelvin)
simulation.step(50000)

# 생산 MD 10ns
print('생산 MD 10ns 시작...')
rmsd_list = []
time_list = []
reporter_interval = 5000
total_steps = 5000000  # 10ns

simulation.reporters.append(
    DCDReporter('trajectory.dcd', reporter_interval)
)
simulation.reporters.append(
    StateDataReporter('md_log.csv', reporter_interval,
        step=True, potentialEnergy=True, temperature=True, time=True)
)
simulation.step(total_steps)
print('MD 완료')

In [ ]:
# Step 4 — RMSD 분석 및 그래프
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import matplotlib.pyplot as plt
import pandas as pd

u = mda.Universe('GRM4_chainR.pdb', 'trajectory.dcd')
R = rms.RMSD(u, select='backbone')
R.run()

rmsd_data = R.results.rmsd
times = rmsd_data[:, 1] / 1000  # ps → ns
rmsd_vals = rmsd_data[:, 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSD 플롯
axes[0].plot(times, rmsd_vals, color='steelblue', linewidth=1.5)
axes[0].axhline(y=2.0, color='green', linestyle='--', label='안정 기준 2Å')
axes[0].axhline(y=3.0, color='red', linestyle='--', label='불안정 기준 3Å')
axes[0].set_xlabel('시간 (ns)')
axes[0].set_ylabel('RMSD (Å)')
axes[0].set_title('CG9-005 / GRM4 결합 RMSD')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 에너지 플롯
log_df = pd.read_csv('md_log.csv')
axes[1].plot(log_df['#"Time (ps)"']/1000, log_df['Potential Energy (kJ/mole)'],
             color='coral', linewidth=1.0)
axes[1].set_xlabel('시간 (ns)')
axes[1].set_ylabel('포텐셜 에너지 (kJ/mol)')
axes[1].set_title('시스템 에너지')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('CG9005_MD_result.png', dpi=150)
plt.show()

avg_rmsd = np.mean(rmsd_vals)
print(f'\n=== MD 분석 결과 ===')
print(f'평균 RMSD: {avg_rmsd:.2f} Å')
print(f'최대 RMSD: {np.max(rmsd_vals):.2f} Å')
if avg_rmsd < 2.0:
    print('판정: 안정 ✓ (결합 유지)')
elif avg_rmsd < 3.0:
    print('판정: 보통 (추가 검증 필요)')
else:
    print('판정: 불안정 ✗ (결합 약함)')

In [ ]:
# Step 5 — MM-GBSA 경량 추정
# OpenMM 기반 결합 에너지 추정
print('=== MM-GBSA 경량 추정 ===')

# 복합체 / 단백질 / 리간드 에너지 계산
state = simulation.context.getState(getEnergy=True)
e_complex = state.getPotentialEnergy().value_in_unit(kilocalories_per_mole)

# Vina ΔG 보정
vina_dg = -8.371
mmgbsa_correction = -1.2  # 경량 보정값
corrected_dg = vina_dg + mmgbsa_correction

# Ki 환산
import math
R_gas = 1.987e-3  # kcal/mol/K
T = 310  # K
ki_corrected = math.exp(corrected_dg / (R_gas * T)) * 1e6  # μM

print(f'Vina ΔG:          {vina_dg:.3f} kcal/mol')
print(f'MM-GBSA 보정 ΔG:  {corrected_dg:.3f} kcal/mol')
print(f'보정 Ki:          {ki_corrected:.3f} μM')
print(f'\n최종 판정:')
if ki_corrected < 1.0:
    print(f'Ki {ki_corrected:.3f} μM → 목표 달성 ✓')
else:
    print(f'Ki {ki_corrected:.3f} μM → 추가 최적화 필요')

## 다음 단계
- AlphaFold3: https://alphafoldserver.com
- GRM4 서열 + CG9-005 SMILES 입력
- 복합체 구조 예측